# Text Flappy Bird: Monte Carlo vs Sarsa(lambda)

**Name:** Massine El Khader  


## 1. Setup

Figures are saved to `figures/`. Adjust `N_MAIN` and `N_SWEEP` to trade off run time vs result quality.

In [1]:
import gymnasium as gym
import text_flappy_bird_gym 

from env_utils import make_env, get_state, DEFAULT_MAX_EPISODE_STEPS
from agents.mc_agent import MCAgent
from agents.sarsa_lambda import SarsaLambdaAgent

import random

In [2]:
import copy
import os
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from utils import evaluate, smooth

ROOT = Path.cwd()
FIGURES_DIR = ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_ENV = dict(height=15, width=20, pipe_gap=4)
RANDOM_SEED = 7
FAST_MODE = True
MAX_EPISODE_STEPS = DEFAULT_MAX_EPISODE_STEPS

if FAST_MODE:
    N_MAIN = 6000
    N_SWEEP = 1500
    N_EVAL = 200
    CHECKPOINTS = [500, 1000, 3000, 6000]
    SMOOTH_WINDOW = 120
else:
    N_MAIN = 8000
    N_SWEEP = 2500
    N_EVAL = 300
    CHECKPOINTS = [500, 1000, 2000, 4000, 8000]
    SMOOTH_WINDOW = 150

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RESULTS = {}


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def savefig(name: str):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"saved {path}")


def _fmt(value, digits=2):
    if value is None:
        return "--"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return f"{float(value):.{digits}f}"


def train_mc(n_episodes=N_MAIN, **kwargs):
    seed = kwargs.pop("seed", None)
    if seed is not None:
        set_global_seed(seed)
    episode_step_limit = kwargs.pop("max_episode_steps", MAX_EPISODE_STEPS)
    env = make_env(
        **{**DEFAULT_ENV, **kwargs.pop("env_kwargs", {})},
        max_episode_steps=episode_step_limit,
    )
    show_progress = kwargs.pop("show_progress", False)
    progress_desc = kwargs.pop("progress_desc", "MC training")
    agent = MCAgent(**kwargs)
    rewards = agent.train(
        env,
        n_episodes=n_episodes,
        show_progress=show_progress,
        progress_desc=progress_desc,
        max_steps_per_episode=episode_step_limit,
    )
    env.close()
    return agent, rewards


def train_sarsa(n_episodes=N_MAIN, **kwargs):
    seed = kwargs.pop("seed", None)
    if seed is not None:
        set_global_seed(seed)
    episode_step_limit = kwargs.pop("max_episode_steps", MAX_EPISODE_STEPS)
    env = make_env(
        **{**DEFAULT_ENV, **kwargs.pop("env_kwargs", {})},
        max_episode_steps=episode_step_limit,
    )
    show_progress = kwargs.pop("show_progress", False)
    progress_desc = kwargs.pop("progress_desc", "Sarsa(lambda) training")
    agent = SarsaLambdaAgent(**kwargs)
    rewards = agent.train(
        env,
        n_episodes=n_episodes,
        show_progress=show_progress,
        progress_desc=progress_desc,
        max_steps_per_episode=episode_step_limit,
    )
    env.close()
    return agent, rewards


def evaluate_agent(
    agent,
    env_kwargs=None,
    n_episodes=N_EVAL,
    show_progress=False,
    progress_desc="Evaluation",
):
    env = make_env(
        **{**DEFAULT_ENV, **(env_kwargs or {})},
        max_episode_steps=MAX_EPISODE_STEPS,
    )
    mean_reward, mean_score = evaluate(
        agent,
        env,
        n_episodes=n_episodes,
        show_progress=show_progress,
        progress_desc=progress_desc,
        max_steps_per_episode=MAX_EPISODE_STEPS,
    )
    env.close()
    return mean_reward, mean_score


def episodes_to_threshold(rewards, threshold):
    for idx, value in enumerate(smooth(rewards, window=SMOOTH_WINDOW), start=SMOOTH_WINDOW):
        if value >= threshold:
            return idx
    return None


def q_table_to_grid(q_table):
    states = list(q_table.keys())
    xs = sorted({s[0] for s in states})
    ys = sorted({s[1] for s in states})
    grid = np.zeros((len(ys), len(xs)))
    for i, dy in enumerate(ys):
        for j, dx in enumerate(xs):
            q = q_table.get((dx, dy), [0.0, 0.0])
            grid[i, j] = q[1] - q[0]
    return grid, xs, ys


def value_table_to_grid(q_table):
    states = list(q_table.keys())
    xs = sorted({s[0] for s in states})
    ys = sorted({s[1] for s in states})
    grid = np.zeros((len(ys), len(xs)))
    for i, dy in enumerate(ys):
        for j, dx in enumerate(xs):
            q = q_table.get((dx, dy), [0.0, 0.0])
            grid[i, j] = max(q)
    return grid, xs, ys


def plot_q_heatmap(q_table, title, filename=None):
    grid, xs, ys = q_table_to_grid(q_table)
    plt.figure(figsize=(9, 5))
    image = plt.imshow(grid, aspect="auto", origin="lower", cmap="RdYlGn")
    plt.colorbar(image, label="Q(flap) - Q(noop)")
    plt.xticks(range(len(xs)), xs, fontsize=7)
    plt.yticks(range(len(ys)), ys, fontsize=7)
    plt.xlabel("dx (horizontal distance to gap)")
    plt.ylabel("dy (vertical distance to gap)")
    plt.title(title)
    plt.tight_layout()
    if filename:
        savefig(filename)


def plot_value_heatmap(q_table, title, filename=None):
    grid, xs, ys = value_table_to_grid(q_table)
    plt.figure(figsize=(9, 5))
    image = plt.imshow(grid, aspect="auto", origin="lower", cmap="viridis")
    plt.colorbar(image, label="V(s) = max_a Q(s, a)")
    plt.xticks(range(len(xs)), xs, fontsize=7)
    plt.yticks(range(len(ys)), ys, fontsize=7)
    plt.xlabel("dx (horizontal distance to gap)")
    plt.ylabel("dy (vertical distance to gap)")
    plt.title(title)
    plt.tight_layout()
    if filename:
        savefig(filename)


def summarize_agent(name, agent, rewards):
    mean_reward, mean_score = evaluate_agent(
        agent,
        show_progress=True,
        progress_desc=f"{name} evaluation",
    )
    print(
        f"{name:<12} reward={mean_reward:>8.2f}  score={mean_score:>8.2f}  "
        f"visited_states={len(agent.Q):>6}"
    )
    return mean_reward, mean_score


def print_timing(label: str, start_time: float):
    elapsed = time.time() - start_time
    print(f"{label} completed in {elapsed / 60:.1f} min ({elapsed:.1f}s)")


def timed_tqdm(iterable, desc):
    return tqdm(
        iterable,
        desc=desc,
        dynamic_ncols=True,
        smoothing=0.1,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
    )


print(
    f"FAST_MODE={FAST_MODE} | N_MAIN={N_MAIN} | N_SWEEP={N_SWEEP} | N_EVAL={N_EVAL} | FIGURES_DIR={FIGURES_DIR}"
)

FAST_MODE=True | N_MAIN=6000 | N_SWEEP=1500 | N_EVAL=200 | FIGURES_DIR=c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures


## 2. Environment Overview

`TextFlappyBird-v0` returns a compact discrete observation `(dx, dy)`:
- `dx`: horizontal distance to the center of the next pipe gap
- `dy`: vertical distance from the bird to that gap center

This makes tabular control feasible. The cell below checks the observation format and builds a random-policy baseline.

In [3]:
env = make_env(**DEFAULT_ENV)
obs, info = env.reset()
print("sample observation:", obs)
print("state tuple:", get_state(obs))
print("action space size:", env.action_space.n)
env.close()

random_rewards = []
random_scores = []
env = make_env(**DEFAULT_ENV)
for _ in timed_tqdm(range(50), desc="Random baseline episodes"):
    obs, info = env.reset()
    done = False
    total_reward = 0
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
    random_rewards.append(total_reward)
    random_scores.append(info.get("score", 0))
env.close()

print(
    f"random baseline over 50 episodes: reward={np.mean(random_rewards):.2f}, "
    f"score={np.mean(random_scores):.2f}"
)


sample observation: (13, -1)
state tuple: (13, -1)
action space size: 2


Random baseline episodes:   0%|          | 0/50 [00:00<?, ?it/s]

random baseline over 50 episodes: reward=10.70, score=0.12


## 3. Compare the Two TFB Environment Versions

The assignment asks for a discussion of the two environment variants. This section inspects both directly:
- `TextFlappyBird-v0` returns the compact `(dx, dy)` state used by the tabular agents
- `TextFlappyBird-screen-v0` returns the full text screen, which is dramatically larger and not suitable for plain tabular RL

In [4]:
compact_env = gym.make("TextFlappyBird-v0", **DEFAULT_ENV)
screen_env = gym.make("TextFlappyBird-screen-v0", **DEFAULT_ENV)

compact_obs, _ = compact_env.reset()
screen_obs, _ = screen_env.reset()

print("compact env observation:", compact_obs, "-> state tuple:", get_state(compact_obs))
print("screen env observation shape:", np.array(screen_obs).shape)
print("screen env dtype:", np.array(screen_obs).dtype)
print("screen env unique tokens in sample frame:", np.unique(np.array(screen_obs))[:20])

compact_env.close()
screen_env.close()


compact env observation: (13, -1) -> state tuple: (13, -1)
screen env observation shape: (20, 15)
screen env dtype: int32
screen env unique tokens in sample frame: [0 1 2]


## 4. Train the Monte Carlo Agent

First-Visit Monte Carlo updates action values only after an episode ends. This is slower early in training but gives a clean non-bootstrapped baseline.

In [5]:
t0 = time.time()
mc_agent, mc_rewards = train_mc(
    n_episodes=N_MAIN,
    epsilon=0.1,
    gamma=0.99,
    show_progress=True,
    progress_desc="Main MC training",
)
mc_mean_reward, mc_mean_score = summarize_agent("MC", mc_agent, mc_rewards)
print_timing("MC training", t0)


Main MC training:   0%|          | 0/6000 [00:00<?, ?it/s]

MC evaluation:   0%|          | 0/200 [00:00<?, ?it/s]

MC           reward= 2000.00  score=  199.00  visited_states=   271
MC training completed in 1.4 min (82.8s)


## 5. Train the Sarsa(lambda) Agent

Sarsa(lambda) updates online every step and uses eligibility traces to propagate credit backward through recent state-action pairs.

In [6]:
t0 = time.time()
sa_agent, sa_rewards = train_sarsa(
    n_episodes=N_MAIN,
    epsilon=0.1,
    alpha=0.1,
    gamma=0.99,
    lambda_=0.8,
    show_progress=True,
    progress_desc="Main Sarsa(lambda) training",
)
sa_mean_reward, sa_mean_score = summarize_agent("Sarsa(lambda)", sa_agent, sa_rewards)
print_timing("Sarsa(lambda) training", t0)


Main Sarsa(lambda) training:   0%|          | 0/6000 [00:00<?, ?it/s]

Sarsa(lambda) evaluation:   0%|          | 0/200 [00:00<?, ?it/s]

Sarsa(lambda) reward= 1039.13  score=  103.00  visited_states=   253
Sarsa(lambda) training completed in 0.4 min (25.9s)


## 6. Main Comparison

Core comparison figures:
- reward curves during training
- final greedy-policy performance

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for rewards, label, color in [
    (mc_rewards, "MC", "steelblue"),
    (sa_rewards, "Sarsa(lambda)", "darkorange"),
]:
    smoothed = smooth(rewards, window=SMOOTH_WINDOW)
    xs = range(len(rewards) - len(smoothed), len(rewards))
    axes[0].plot(xs, smoothed, label=label, color=color)

axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Smoothed total reward")
axes[0].set_title("Learning curves")
axes[0].legend()

bars = axes[1].bar(
    ["MC", "Sarsa(lambda)"],
    [mc_mean_score, sa_mean_score],
    color=["steelblue", "darkorange"],
)
axes[1].set_ylabel(f"Mean score over {N_EVAL} greedy episodes")
axes[1].set_title("Final policy performance")
for bar in bars:
    value = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 0.1, f"{value:.2f}", ha="center")

plt.tight_layout()
savefig("fig_reward_curves.png")

RESULTS["main"] = {
    "mc_reward": mc_mean_reward,
    "mc_score": mc_mean_score,
    "sarsa_reward": sa_mean_reward,
    "sarsa_score": sa_mean_score,
    "mc_states": len(mc_agent.Q),
    "sarsa_states": len(sa_agent.Q),
}

print("Main comparison summary:")
print(f"{'Agent':<16}{'Mean reward':>14}{'Mean score':>14}{'Visited states':>18}")
print(f"{'MC':<16}{mc_mean_reward:>14.2f}{mc_mean_score:>14.2f}{len(mc_agent.Q):>18}")
print(f"{'Sarsa(lambda)':<16}{sa_mean_reward:>14.2f}{sa_mean_score:>14.2f}{len(sa_agent.Q):>18}")


saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_reward_curves.png
Main comparison summary:
Agent              Mean reward    Mean score    Visited states
MC                     2000.00        199.00               271
Sarsa(lambda)          1039.13        103.00               253


## 7. Action-Value Visualisation

The heatmaps below show both:
- action preference through `Q(flap) - Q(noop)`
- state value through `V(s) = max_a Q(s, a)`

That gives explicit value-function plots as well as action-value interpretation.

In [8]:
plot_q_heatmap(mc_agent.Q, "Monte Carlo: Q(flap) - Q(noop)", "fig_q_heatmap_mc.png")
plot_q_heatmap(sa_agent.Q, "Sarsa(lambda): Q(flap) - Q(noop)", "fig_q_heatmap_sarsa.png")
plot_value_heatmap(mc_agent.Q, "Monte Carlo: V(s) = max_a Q(s, a)", "fig_v_heatmap_mc.png")
plot_value_heatmap(sa_agent.Q, "Sarsa(lambda): V(s) = max_a Q(s, a)", "fig_v_heatmap_sarsa.png")


saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_q_heatmap_mc.png
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_q_heatmap_sarsa.png
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_v_heatmap_mc.png
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_v_heatmap_sarsa.png


## 8. Policy Evolution During Training

To show how the learned decision boundary develops, train a fresh Sarsa(lambda) agent in chunks and snapshot its Q-table at several checkpoints.

In [9]:
evolution_agent = SarsaLambdaAgent(epsilon=0.1, alpha=0.1, gamma=0.99, lambda_=0.8)
evolution_env = make_env(**DEFAULT_ENV)

snapshots = {}
trained_so_far = 0
evolution_start = time.time()
for checkpoint in timed_tqdm(CHECKPOINTS, desc="Policy evolution checkpoints"):
    evolution_agent.train(
        evolution_env,
        n_episodes=checkpoint - trained_so_far,
        show_progress=True,
        progress_desc=f"Policy evolution to {checkpoint}",
    )
    trained_so_far = checkpoint
    snapshots[checkpoint] = copy.deepcopy(dict(evolution_agent.Q))

evolution_env.close()

fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(4 * len(CHECKPOINTS), 4))
if len(CHECKPOINTS) == 1:
    axes = [axes]

for ax, checkpoint in zip(axes, CHECKPOINTS):
    grid, xs, ys = q_table_to_grid(snapshots[checkpoint])
    image = ax.imshow(grid, aspect="auto", origin="lower", cmap="RdYlGn")
    ax.set_title(f"{checkpoint} episodes")
    ax.set_xlabel("dx")
    ax.set_ylabel("dy")
    ax.set_xticks(range(0, len(xs), max(1, len(xs) // 6)))
    ax.set_xticklabels([xs[i] for i in range(0, len(xs), max(1, len(xs) // 6))], fontsize=7)
    ax.set_yticks(range(0, len(ys), max(1, len(ys) // 6)))
    ax.set_yticklabels([ys[i] for i in range(0, len(ys), max(1, len(ys) // 6))], fontsize=7)

fig.colorbar(image, ax=axes, shrink=0.8, label="Q(flap) - Q(noop)")
plt.tight_layout()
savefig("fig_policy_evolution.png")
print_timing("Policy evolution analysis", evolution_start)


Policy evolution checkpoints:   0%|          | 0/4 [00:00<?, ?it/s]

Policy evolution to 500:   0%|          | 0/500 [00:00<?, ?it/s]

Policy evolution to 1000:   0%|          | 0/500 [00:00<?, ?it/s]

Policy evolution to 3000:   0%|          | 0/2000 [00:00<?, ?it/s]

Policy evolution to 6000:   0%|          | 0/3000 [00:00<?, ?it/s]

C:\Users\pcc\AppData\Local\Temp\ipykernel_23292\1266214544.py:35: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_policy_evolution.png
Policy evolution analysis completed in 0.6 min (36.6s)


## 9. Convergence Speed Analysis

Compare how quickly each method reaches smoothed reward thresholds and visualise the gap directly.

In [10]:
thresholds = [5, 10, 15, 20, 25]
mc_hits = [episodes_to_threshold(mc_rewards, t) for t in thresholds]
sa_hits = [episodes_to_threshold(sa_rewards, t) for t in thresholds]

print(f"{'Threshold':<12}{'MC episodes':>14}{'Sarsa episodes':>18}")
for threshold, mc_hit, sa_hit in zip(thresholds, mc_hits, sa_hits):
    print(f"{threshold:<12}{str(mc_hit):>14}{str(sa_hit):>18}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for rewards, label, color in [
    (mc_rewards, "MC", "steelblue"),
    (sa_rewards, "Sarsa(lambda)", "darkorange"),
]:
    smoothed = smooth(rewards, window=SMOOTH_WINDOW)
    xs = range(len(rewards) - len(smoothed), len(rewards))
    axes[0].plot(xs, smoothed, label=label, color=color)
for threshold in thresholds:
    axes[0].axhline(threshold, linestyle="--", linewidth=0.8, color="gray", alpha=0.4)
axes[0].set_title("Learning curves with reward thresholds")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Smoothed total reward")
axes[0].legend()

x = np.arange(len(thresholds))
width = 0.36
mc_bars = [v if v is not None else N_MAIN for v in mc_hits]
sa_bars = [v if v is not None else N_MAIN for v in sa_hits]
axes[1].bar(x - width / 2, mc_bars, width, label="MC", color="steelblue")
axes[1].bar(x + width / 2, sa_bars, width, label="Sarsa(lambda)", color="darkorange")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in thresholds])
axes[1].set_xlabel("Reward threshold")
axes[1].set_ylabel("Episodes needed")
axes[1].set_title("Convergence speed")
axes[1].legend()

plt.tight_layout()
savefig("fig_convergence.png")

RESULTS["convergence"] = {
    "thresholds": thresholds,
    "mc": mc_hits,
    "sarsa": sa_hits,
}


Threshold      MC episodes    Sarsa episodes


5                      120               156
10                     189              2295
15                     246              3165
20                     291              4256
25                     313              4377
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_convergence.png


## 10. Parameter Sweeps

The assignment explicitly asks for parameter sensitivity. The next cell runs:
- epsilon sweep for Monte Carlo
- epsilon sweep for Sarsa(lambda)
- alpha sweep for Sarsa(lambda)
- lambda sweep for Sarsa(lambda)
- gamma sweep for Sarsa(lambda)

In [11]:
epsilon_values = [0.01, 0.05, 0.1, 0.2, 0.4]
alpha_values = [0.01, 0.05, 0.1, 0.2, 0.4]
lambda_values = [0.0, 0.3, 0.5, 0.8, 0.95, 1.0]
gamma_values = [0.9, 0.95, 0.99, 1.0]

def run_sweep(agent_name, values, build_agent, key_name):
    records = []
    sweep_start = time.time()
    for value in timed_tqdm(values, desc=f"{agent_name} {key_name} sweep"):
        env = make_env(**DEFAULT_ENV)
        agent = build_agent(value)
        rewards = agent.train(
            env,
            n_episodes=N_SWEEP,
            show_progress=True,
            progress_desc=f"{agent_name} {key_name}={value}",
        )
        env.close()
        mean_reward, mean_score = evaluate_agent(
            agent,
            show_progress=True,
            progress_desc=f"{agent_name} {key_name}={value} eval",
        )
        records.append(
            {
                key_name: value,
                "reward_curve": rewards,
                "mean_reward": mean_reward,
                "score": mean_score,
            }
        )
        print(
            f"{agent_name:<16} {key_name}={value:<6} "
            f"reward={mean_reward:>8.2f} score={mean_score:>8.2f}"
        )
    print_timing(f"{agent_name} {key_name} sweep", sweep_start)
    return records


mc_eps_res = run_sweep("MC", epsilon_values, lambda v: MCAgent(epsilon=v, gamma=0.99), "epsilon")
sa_eps_res = run_sweep(
    "Sarsa(lambda)",
    epsilon_values,
    lambda v: SarsaLambdaAgent(epsilon=v, alpha=0.1, gamma=0.99, lambda_=0.8),
    "epsilon",
)
sa_alpha_res = run_sweep(
    "Sarsa(lambda)",
    alpha_values,
    lambda v: SarsaLambdaAgent(epsilon=0.1, alpha=v, gamma=0.99, lambda_=0.8),
    "alpha",
)
sa_lambda_res = run_sweep(
    "Sarsa(lambda)",
    lambda_values,
    lambda v: SarsaLambdaAgent(epsilon=0.1, alpha=0.1, gamma=0.99, lambda_=v),
    "lambda_",
)
sa_gamma_res = run_sweep(
    "Sarsa(lambda)",
    gamma_values,
    lambda v: SarsaLambdaAgent(epsilon=0.1, alpha=0.1, gamma=v, lambda_=0.8),
    "gamma",
)

RESULTS["sweeps"] = {
    "mc_epsilon": mc_eps_res,
    "sarsa_epsilon": sa_eps_res,
    "sarsa_alpha": sa_alpha_res,
    "sarsa_lambda": sa_lambda_res,
    "sarsa_gamma": sa_gamma_res,
}


MC epsilon sweep:   0%|          | 0/5 [00:00<?, ?it/s]

MC epsilon=0.01:   0%|          | 0/1500 [00:00<?, ?it/s]

MC epsilon=0.01 eval:   0%|          | 0/200 [00:00<?, ?it/s]

MC               epsilon=0.01   reward=   11.45 score=    0.41


MC epsilon=0.05:   0%|          | 0/1500 [00:00<?, ?it/s]

...

Sarsa(lambda) gamma=0.99 eval:   0%|          | 0/200 [00:00<?, ?it/s]

Sarsa(lambda)    gamma=0.99   reward=   12.19 score=    0.24


Sarsa(lambda) gamma=1.0:   0%|          | 0/1500 [00:00<?, ?it/s]

Sarsa(lambda) gamma=1.0 eval:   0%|          | 0/200 [00:00<?, ?it/s]

Sarsa(lambda)    gamma=1.0    reward=   11.15 score=    0.00
Sarsa(lambda) gamma sweep completed in 0.0 min (0.9s)


In [12]:
def draw_sweep(ax, records, key_name, title):
    for record in records:
        rewards = record["reward_curve"]
        smoothed = smooth(rewards, window=max(30, min(SMOOTH_WINDOW, len(rewards) // 3)))
        xs = range(len(rewards) - len(smoothed), len(rewards))
        ax.plot(xs, smoothed, label=f"{key_name}={record[key_name]}")
    ax.set_title(title)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Smoothed reward")
    ax.legend(fontsize=8)


fig, axes = plt.subplots(2, 3, figsize=(17, 9))
draw_sweep(axes[0, 0], mc_eps_res, "epsilon", "MC epsilon sweep")
draw_sweep(axes[0, 1], sa_eps_res, "epsilon", "Sarsa(lambda) epsilon sweep")
draw_sweep(axes[0, 2], sa_alpha_res, "alpha", "Sarsa(lambda) alpha sweep")
draw_sweep(axes[1, 0], sa_lambda_res, "lambda_", "Sarsa(lambda) lambda sweep")
draw_sweep(axes[1, 1], sa_gamma_res, "gamma", "Sarsa(lambda) gamma sweep")

best_scores = [
    max(mc_eps_res, key=lambda r: r["score"])["score"],
    max(sa_eps_res, key=lambda r: r["score"])["score"],
    max(sa_alpha_res, key=lambda r: r["score"])["score"],
    max(sa_lambda_res, key=lambda r: r["score"])["score"],
    max(sa_gamma_res, key=lambda r: r["score"])["score"],
]
names = ["MC eps", "Sarsa eps", "Sarsa alpha", "Sarsa lambda", "Sarsa gamma"]
axes[1, 2].bar(names, best_scores, color=["steelblue", "darkorange", "darkorange", "darkorange", "darkorange"])
axes[1, 2].set_title("Best score from each sweep")
axes[1, 2].set_ylabel("Greedy evaluation score")
axes[1, 2].tick_params(axis="x", rotation=20)

plt.tight_layout()
savefig("fig_param_sweep.png")

print("Best parameters from sweeps:")
print("MC epsilon:", max(mc_eps_res, key=lambda r: r["score"])["epsilon"])
print("Sarsa epsilon:", max(sa_eps_res, key=lambda r: r["score"])["epsilon"])
print("Sarsa alpha:", max(sa_alpha_res, key=lambda r: r["score"])["alpha"])
print("Sarsa lambda:", max(sa_lambda_res, key=lambda r: r["score"])["lambda_"])
print("Sarsa gamma:", max(sa_gamma_res, key=lambda r: r["score"])["gamma"])


saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_param_sweep.png
Best parameters from sweeps:
MC epsilon: 0.2
Sarsa epsilon: 0.4
Sarsa alpha: 0.4
Sarsa lambda: 0.8
Sarsa gamma: 0.99


## 11. Robustness Across Random Seeds

A single run can be noisy in reinforcement learning. This section repeats the main training experiment across several seeds and summarises the distribution of final scores.

In [13]:
ROBUSTNESS_SEEDS = [3, 7, 21]
robust_rows = []
robustness_start = time.time()
approx_train_count = len(ROBUSTNESS_SEEDS) * 2
approx_episode_count = approx_train_count * max(3000, N_SWEEP)
print(
    f"Robustness plan: {len(ROBUSTNESS_SEEDS)} seeds, {approx_train_count} trainings, "
    f"about {approx_episode_count} episodes total."
)

for seed in timed_tqdm(ROBUSTNESS_SEEDS, desc="Robustness seeds"):
    print(f"running seed {seed}")
    mc_tmp, _ = train_mc(
        n_episodes=max(3000, N_SWEEP),
        epsilon=0.1,
        gamma=0.99,
        seed=seed,
        show_progress=True,
        progress_desc=f"Robustness MC seed={seed}",
    )
    sa_tmp, _ = train_sarsa(
        n_episodes=max(3000, N_SWEEP),
        epsilon=0.1,
        alpha=0.1,
        gamma=0.99,
        lambda_=0.8,
        seed=seed,
        show_progress=True,
        progress_desc=f"Robustness Sarsa seed={seed}",
    )
    mc_r, mc_s = evaluate_agent(
        mc_tmp,
        show_progress=True,
        progress_desc=f"Robustness MC seed={seed} eval",
    )
    sa_r, sa_s = evaluate_agent(
        sa_tmp,
        show_progress=True,
        progress_desc=f"Robustness Sarsa seed={seed} eval",
    )
    robust_rows.append(
        {
            "seed": seed,
            "mc_reward": mc_r,
            "mc_score": mc_s,
            "sarsa_reward": sa_r,
            "sarsa_score": sa_s,
        }
    )

print(f"{'Seed':<8}{'MC score':>12}{'Sarsa score':>15}")
for row in robust_rows:
    print(f"{row['seed']:<8}{row['mc_score']:>12.2f}{row['sarsa_score']:>15.2f}")

mc_seed_scores = [row["mc_score"] for row in robust_rows]
sa_seed_scores = [row["sarsa_score"] for row in robust_rows]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(ROBUSTNESS_SEEDS))
width = 0.36
ax.bar(x - width / 2, mc_seed_scores, width, color="steelblue", label="MC")
ax.bar(x + width / 2, sa_seed_scores, width, color="darkorange", label="Sarsa(lambda)")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in ROBUSTNESS_SEEDS])
ax.set_xlabel("Random seed")
ax.set_ylabel(f"Mean score over {N_EVAL} greedy episodes")
ax.set_title("Robustness across seeds")
ax.legend()
plt.tight_layout()
savefig("fig_multiseed_comparison.png")

RESULTS["multiseed"] = robust_rows
print(
    f"MC score mean±std: {np.mean(mc_seed_scores):.2f} ± {np.std(mc_seed_scores):.2f} | "
    f"Sarsa score mean±std: {np.mean(sa_seed_scores):.2f} ± {np.std(sa_seed_scores):.2f}"
)
print_timing("Robustness analysis", robustness_start)


Robustness plan: 3 seeds, 6 trainings, about 18000 episodes total.


Robustness seeds:   0%|          | 0/3 [00:00<?, ?it/s]

running seed 3


Robustness MC seed=3:   0%|          | 0/3000 [00:00<?, ?it/s]

Robustness Sarsa seed=3:   0%|          | 0/3000 [00:00<?, ?it/s]

...

Robustness MC seed=21:   0%|          | 0/3000 [00:00<?, ?it/s]

Robustness Sarsa seed=21:   0%|          | 0/3000 [00:00<?, ?it/s]

Robustness MC seed=21 eval:   0%|          | 0/200 [00:00<?, ?it/s]

Robustness Sarsa seed=21 eval:   0%|          | 0/200 [00:00<?, ?it/s]

Seed        MC score    Sarsa score
3             199.00           0.65
7             199.00           0.00
21            199.00           0.64
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_multiseed_comparison.png
MC score mean±std: 199.00 ± 0.00 | Sarsa score mean±std: 0.43 ± 0.30
Robustness analysis completed in 1.4 min (82.4s)


## 12. Generalisation Across Environment Configurations

Train on the default configuration and evaluate the learned policy on different level settings. This directly answers the transfer question from the assignment.

In [14]:
generalization_configs = [
    ("train config\nh=15,w=20,gap=4", dict(height=15, width=20, pipe_gap=4)),
    ("gap=2\n(harder)", dict(height=15, width=20, pipe_gap=2)),
    ("gap=3", dict(height=15, width=20, pipe_gap=3)),
    ("gap=6\n(easier)", dict(height=15, width=20, pipe_gap=6)),
    ("gap=8", dict(height=15, width=20, pipe_gap=8)),
    ("h=10,w=15\ngap=4", dict(height=10, width=15, pipe_gap=4)),
    ("h=20,w=25\ngap=4", dict(height=20, width=25, pipe_gap=4)),
]

gen_labels = []
mc_scores_gen = []
sa_scores_gen = []
generalization_start = time.time()

for label, cfg in timed_tqdm(generalization_configs, desc="Generalization configs"):
    gen_labels.append(label)
    _, mc_score = evaluate_agent(
        mc_agent,
        env_kwargs=cfg,
        show_progress=True,
        progress_desc=f"MC generalization {label.replace(chr(10), ' ')}",
    )
    _, sa_score = evaluate_agent(
        sa_agent,
        env_kwargs=cfg,
        show_progress=True,
        progress_desc=f"Sarsa generalization {label.replace(chr(10), ' ')}",
    )
    mc_scores_gen.append(mc_score)
    sa_scores_gen.append(sa_score)
    print(f"{label.replace(chr(10), ' '):<28} MC={mc_score:>8.2f}  Sarsa={sa_score:>8.2f}")

x = np.arange(len(gen_labels))
width = 0.36
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(x - width / 2, mc_scores_gen, width, label="MC", color="steelblue")
ax.bar(x + width / 2, sa_scores_gen, width, label="Sarsa(lambda)", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(gen_labels)
ax.set_ylabel(f"Mean score over {N_EVAL} evaluation episodes")
ax.set_title("Generalisation to new Text Flappy Bird configurations")
ax.legend()
plt.tight_layout()
savefig("fig_generalization.png")

RESULTS["generalization"] = [
    {"label": label, "mc_score": mc_score, "sarsa_score": sa_score}
    for label, mc_score, sa_score in zip(gen_labels, mc_scores_gen, sa_scores_gen)
]
print_timing("Generalization evaluation", generalization_start)


Generalization configs:   0%|          | 0/7 [00:00<?, ?it/s]

MC generalization train config h=15,w=20,gap=4:   0%|          | 0/200 [00:00<?, ?it/s]

Sarsa generalization train config h=15,w=20,gap=4:   0%|          | 0/200 [00:00<?, ?it/s]

train config h=15,w=20,gap=4 MC=  199.00  Sarsa=  100.23


MC generalization gap=2 (harder):   0%|          | 0/200 [00:00<?, ?it/s]

...

Sarsa generalization h=10,w=15 gap=4:   0%|          | 0/200 [00:00<?, ?it/s]

h=10,w=15 gap=4              MC=   77.98  Sarsa=  249.00


MC generalization h=20,w=25 gap=4:   0%|          | 0/200 [00:00<?, ?it/s]

Sarsa generalization h=20,w=25 gap=4:   0%|          | 0/200 [00:00<?, ?it/s]

h=20,w=25 gap=4              MC=    0.00  Sarsa=    0.00
saved c:\Users\pcc\Desktop\CS\3A\SDI\SM11\RL\individual assignment\figures\fig_generalization.png
Generalization evaluation completed in 1.5 min (90.8s)


## 13. Results Summary

The next cell prints a compact summary of the key results.

In [15]:
mc_conv = episodes_to_threshold(mc_rewards, 10)
sa_conv = episodes_to_threshold(sa_rewards, 10)

print("Report summary")
print("==============")
print(f"Main setup: TextFlappyBird-v0 with {DEFAULT_ENV}")
print(f"MC final reward={mc_mean_reward:.2f}, score={mc_mean_score:.2f}, visited states={len(mc_agent.Q)}")
print(f"Sarsa final reward={sa_mean_reward:.2f}, score={sa_mean_score:.2f}, visited states={len(sa_agent.Q)}")
print(f"Episodes to reach smoothed reward >= 10: MC={mc_conv}, Sarsa={sa_conv}")
if mc_conv and sa_conv:
    print(f"Sarsa speedup at threshold 10: {mc_conv / sa_conv:.2f}x")

print()
print("Environment comparison:")
print("- TextFlappyBird-v0 returns compact (dx, dy) features, so tabular RL is feasible.")
print("- TextFlappyBird-screen-v0 returns the full rendered screen, which makes the state space too large for tabular methods.")
print("- The compact environment is easier to solve but hides velocity and full-screen context.")

if "multiseed" in RESULTS:
    mc_seed_scores = [row["mc_score"] for row in RESULTS["multiseed"]]
    sa_seed_scores = [row["sarsa_score"] for row in RESULTS["multiseed"]]
    print()
    print(
        f"Robustness across seeds: MC={np.mean(mc_seed_scores):.2f}±{np.std(mc_seed_scores):.2f}, "
        f"Sarsa={np.mean(sa_seed_scores):.2f}±{np.std(sa_seed_scores):.2f}"
    )

print()
print("Original Flappy Bird applicability:")
print("- The same tabular agents are not suitable as-is for raw pixel observations.")
print("- Sarsa(lambda) could be adapted with function approximation or deep RL.")
print("- Monte Carlo control becomes sample-inefficient in continuous or image-based state spaces.")

print()
print("Best sweep settings from this notebook:")
print(f"- MC best epsilon: {max(mc_eps_res, key=lambda r: r['score'])['epsilon']}")
print(f"- Sarsa best epsilon: {max(sa_eps_res, key=lambda r: r['score'])['epsilon']}")
print(f"- Sarsa best alpha: {max(sa_alpha_res, key=lambda r: r['score'])['alpha']}")
print(f"- Sarsa best lambda: {max(sa_lambda_res, key=lambda r: r['score'])['lambda_']}")
print(f"- Sarsa best gamma: {max(sa_gamma_res, key=lambda r: r['score'])['gamma']}")


Report summary
Main setup: TextFlappyBird-v0 with {'height': 15, 'width': 20, 'pipe_gap': 4}
MC final reward=2000.00, score=199.00, visited states=337
Sarsa final reward=1039.13, score=103.00, visited states=318
Episodes to reach smoothed reward >= 10: MC=189, Sarsa=2295
Sarsa speedup at threshold 10: 0.08x

Environment comparison:
- TextFlappyBird-v0 returns compact (dx, dy) features, so tabular RL is feasible.
- TextFlappyBird-screen-v0 returns the full rendered screen, which makes the state space too large for tabular methods.
- The compact environment is easier to solve but hides velocity and full-screen context.

Robustness across seeds: MC=199.00±0.00, Sarsa=0.43±0.30

Original Flappy Bird applicability:
- The same tabular agents are not suitable as-is for raw pixel observations.
- Sarsa(lambda) could be adapted with function approximation or deep RL.
- Monte Carlo control becomes sample-inefficient in continuous or image-based state spaces.

Best sweep settings from this noteboo